Evaluation notebook from the RAG and Graph RAG

MulitHopQA

In [2]:
import os
import json
import re
from typing import List, Type, TypeVar, Optional
from pydantic import BaseModel, Field
from openai import OpenAI

T = TypeVar("T", bound=BaseModel)

# ---------------------------------------------------------------------------
# Robust JSON Parsing Helper
# ---------------------------------------------------------------------------

def parse_json_response(raw_text: str, model_class: Type[T]) -> T:
    """
    Cleans model response text and safely validates it into a Pydantic model.
    Handles extra text, conversational prefixes, and markdown code fences.
    """
    if not raw_text:
        raise ValueError("Received empty response from LLM.")

    text = raw_text.strip()

    # Strip markdown code blocks (e.g. ```json ... ```)
    text = re.sub(r"^```(?:json)?\s*", "", text, flags=re.IGNORECASE)
    text = re.sub(r"\s*```$", "", text)

    # Extract outermost JSON object if model included introductory prose
    json_match = re.search(r"\{.*\}", text, re.DOTALL)
    if json_match:
        text = json_match.group(0)

    try:
        return model_class.model_validate_json(text)
    except Exception as e:
        # Fallback: parse via dict for detailed error context
        data = json.loads(text)
        return model_class.model_validate(data)


# ---------------------------------------------------------------------------
# Pydantic Schemas for Evaluation Outputs
# ---------------------------------------------------------------------------

class ClaimVerification(BaseModel):
    claim: str = Field(description="Extracted factual claim from the answer.")
    is_supported: bool = Field(description="True if the claim is supported by context.")
    evidence: str = Field(description="Supporting sentence from context, or 'None' if unsupported.")

class FaithfulnessAssessment(BaseModel):
    claims: List[ClaimVerification] = Field(description="Extracted claims and their verification status.")
    faithfulness_score: float = Field(description="Ratio of supported claims to total claims (0.0 to 1.0).")
    reasoning: str = Field(description="Summary explanation of groundedness.")

class AnswerRelevanceAssessment(BaseModel):
    relevance_score: float = Field(description="Score between 0.0 (irrelevant) and 1.0 (directly answers query).")
    reasoning: str = Field(description="Explanation of query intent alignment.")

class ContextRelevanceAssessment(BaseModel):
    context_precision_score: float = Field(description="Score between 0.0 (all noise) and 1.0 (highly relevant).")
    useful_snippets: List[str] = Field(description="Key snippets in context that address the query.")
    reasoning: str = Field(description="Explanation of context usefulness and noise levels.")

class SynthesisClaim(BaseModel):
    claim: str = Field(description="Factual claim extracted from source documents.")
    source_doc: str = Field(description="Identifier or reference of the source document.")
    is_supported: bool = Field(description="Whether claim is supported by context.")

class MultiHopRAGReasoningAssessment(BaseModel):
    extracted_claims: List[SynthesisClaim] = Field(description="Claims drawn across documents.")
    category_check_passed: bool = Field(description="Whether category-specific logical check passed.")
    synthesis_score: float = Field(description="Score between 0.0 and 1.0 evaluating cross-document synthesis.")
    reasoning: str = Field(description="Detailed explanation of multi-document evaluation.")

class RAGEvalReport(BaseModel):
    question: str
    answer: str
    context: str
    faithfulness: FaithfulnessAssessment
    answer_relevance: AnswerRelevanceAssessment
    context_relevance: ContextRelevanceAssessment
    composite_score: float = Field(description="Weighted overall score across all metrics.")


# ---------------------------------------------------------------------------
# RAG Evaluator Class
# ---------------------------------------------------------------------------

class RAGEvaluator:
    """Evaluates RAG outputs with schema-enforced structured JSON generation."""

    def __init__(self, model_name: str = "gpt-oss:120b-cloud", api_key: str = "a4284e84d75b4486af315b527679d53c.qybrj67u4A8GEfbPV89KiciS"):
        self.client = OpenAI(base_url="https://ollama.com/v1",api_key="a4284e84d75b4486af315b527679d53c.qybrj67u4A8GEfbPV89KiciS")
        self.model = model_name

        self.system_instruction = (
            "You are a strict evaluation engine. "
            "You MUST output ONLY a valid JSON object matching the requested schema. "
            "Do NOT include conversational text, introductory statements, or markdown formatting."
        )

    def evaluate_faithfulness(self, context: str, answer: str) -> FaithfulnessAssessment:
        """Evaluates whether answer statements are grounded in context."""
        schema = json.dumps(FaithfulnessAssessment.model_json_schema(), indent=2)
        prompt = (
            f"Context:\n{context}\n\n"
            f"Answer:\n{answer}\n\n"
            "Task: Break down the answer into individual factual claims. "
            "Verify whether each claim is supported by the context. "
            "Compute faithfulness_score as (supported_claims / total_claims).\n\n"
            f"Required Output JSON Schema:\n{schema}"
        )
        response = self.client.chat.completions.create(
            model=self.model,
            response_format={"type": "json_object"},
            messages=[
                {"role": "system", "content": self.system_instruction},
                {"role": "user", "content": prompt}
            ],
            temperature=0.0
        )
        return parse_json_response(response.choices[0].message.content, FaithfulnessAssessment)

    def evaluate_cross_doc_synthesis(
        self, 
        query: str, 
        query_category: str, 
        retrieved_docs_text: str, 
        predicted_answer: str
    ) -> MultiHopRAGReasoningAssessment:
        """Evaluates multi-document cross-synthesis and category-specific logic."""
        
        prompt = (
            f"Query: {query}\n"
            f"Query Category: {query_category}\n\n"
            f"Retrieved Document Contexts:\n{retrieved_docs_text}\n\n"
            f"Generated Answer: {predicted_answer}\n\n"
            "Task:\n"
            "1. Identify factual claims drawn from individual source documents.\n"
            "2. Evaluate whether the answer correctly synthesizes evidence across multiple documents.\n"
            "3. Category-specific checks:\n"
            "   - 'temporal': Verify chronological sequence and dates across documents.\n"
            "   - 'comparison': Verify side-by-side metric/attribute alignment.\n"
            "   - 'inference': Verify multi-step logical deduction.\n"
            "   - 'null': Verify whether the model correctly stated that context is insufficient.\n"
            "4. Score multi-document synthesis quality between 0.0 and 1.0."
        )

        response = self.client.beta.chat.completions.parse(
            model=self.model,
            messages=[
                {"role": "system", "content": "You are a specialized MultiHop-RAG evaluator for multi-document synthesis."},
                {"role": "user", "content": prompt}
            ],
            response_format=MultiHopRAGReasoningAssessment,
            temperature=0.0
        )
        return response.choices[0].message.parsed


    def evaluate_answer_relevance(self, question: str, answer: str) -> AnswerRelevanceAssessment:
        """Evaluates whether the answer directly addresses the question."""
        schema = json.dumps(AnswerRelevanceAssessment.model_json_schema(), indent=2)
        prompt = (
            f"Question:\n{question}\n\n"
            f"Answer:\n{answer}\n\n"
            "Task: Score how directly and completely the answer addresses the question intent. "
            "Ignore factual correctness against external facts; focus solely on query intent alignment.\n\n"
            f"Required Output JSON Schema:\n{schema}"
        )
        response = self.client.chat.completions.create(
            model=self.model,
            response_format={"type": "json_object"},
            messages=[
                {"role": "system", "content": self.system_instruction},
                {"role": "user", "content": prompt}
            ],
            temperature=0.0
        )
        return parse_json_response(response.choices[0].message.content, AnswerRelevanceAssessment)

    def evaluate_context_relevance(self, question: str, context: str) -> ContextRelevanceAssessment:
        """Evaluates signal-to-noise ratio in retrieved context."""
        schema = json.dumps(ContextRelevanceAssessment.model_json_schema(), indent=2)
        prompt = (
            f"Question:\n{question}\n\n"
            f"Retrieved Context:\n{context}\n\n"
            "Task: Evaluate how relevant and useful the context is for answering the question. "
            "Extract useful snippets and score precision from 0.0 to 1.0.\n\n"
            f"Required Output JSON Schema:\n{schema}"
        )
        response = self.client.chat.completions.create(
            model=self.model,
            response_format={"type": "json_object"},
            messages=[
                {"role": "system", "content": self.system_instruction},
                {"role": "user", "content": prompt}
            ],
            temperature=0.0
        )
        return parse_json_response(response.choices[0].message.content, ContextRelevanceAssessment)

    def evaluate(
        self, 
        question: str, 
        context: str, 
        answer: str, 
        weights: tuple[float, float, float] = (0.4, 0.4, 0.2)
    ) -> RAGEvalReport:
        """Runs evaluation across all three RAG Triad metrics."""
        faith_eval = self.evaluate_faithfulness(context, answer)
        ans_rel_eval = self.evaluate_answer_relevance(question, answer)
        ctx_rel_eval = self.evaluate_context_relevance(question, context)

        w_f, w_a, w_c = weights
        composite = (
            w_f * faith_eval.faithfulness_score +
            w_a * ans_rel_eval.relevance_score +
            w_c * ctx_rel_eval.context_precision_score
        )

        return RAGEvalReport(
            question=question,
            answer=answer,
            context=context,
            faithfulness=faith_eval,
            answer_relevance=ans_rel_eval,
            context_relevance=ctx_rel_eval,
            composite_score=round(composite, 3)
        )


Question one:

What are all direct incoming and outgoing relationships centered on the Permian Region node?	

In [ ]:
if __name__ == "__main__":
    question = "What are all direct incoming and outgoing relationships centered on the Permian Region node?"
    rag_context = (
        "Appalachia region Bakken region Eagle Ford region Haynesville region Permian region Rest of Lower 48 States"
    )
    rag_answer = (
        "Based on the provided context, which consists of tabular data, there are no explicitly defined graph-based incoming or outgoing structural relationships. However, we can identify the relationships based on the groupings and numerical data (positive and negative flows) associated with the Permian region node. For peer group relationships (sibling nodes), the Permian region belongs to the same category/grouping as the following regional nodes: Bakken region, Eagle Ford region, and Haynesville region."
    )

    evaluator = RAGEvaluator()
    report = evaluator.evaluate(
        question=question,
        context=rag_context,
        answer=rag_answer
    )

    print("--- RAG Evaluation Report ---")
    print(f"Overall RAG Score: {report.composite_score} / 1.0\n")
    print(f"Faithfulness Score: {report.faithfulness.faithfulness_score}")
    print(f"  Reasoning: {report.faithfulness.reasoning}")
    print("  Claims Breakdown:")
    for c in report.faithfulness.claims:
        status = "✓ Supported" if c.is_supported else "✗ Unsupported"
        print(f"    - [{status}] Claim: '{c.claim}' | Evidence: '{c.evidence}'")

    print(f"\nAnswer Relevance Score: {report.answer_relevance.relevance_score}")
    print(f"  Reasoning: {report.answer_relevance.reasoning}")

    print(f"\nContext Precision Score: {report.context_relevance.context_precision_score}")
    print(f"  Reasoning: {report.context_relevance.reasoning}")


    question = "What are all direct incoming and outgoing relationships centered on the Permian Region node?"
    graph_context = (
        "The Permian Region is located in the United States. The Permian Region is located in the Lower 48 States. Cumulative Drilled But Uncompleted Wells are located in the Permian Region. New Wells Drilled are measured in the Permian Region. New Wells Drilled Per Rig are measured in the Permian Region. The Permian Region affects the supply of Total Primary Supply. The U.S. Energy Information Administration forecasts the outlook for the Permian Region. Crude Oil is produced in the Permian Region. Crude Oil Production From Newly Completed Wells is produced in the Permian Region. The Permian Region produces Natural Gas."
    )
    graph_answer = (
        "The direct incoming and outgoing relationships centered on the Permian Region node are outgoing relationships (from Permian Region), including LOCATED_IN United States and Lower 48 States, AFFECTS_SUPPLY_OF Total Primary Supply, and PRODUCES Natural Gas. Incoming relationships (to Permian Region) include Cumulative Drilled But Uncompleted Wells is LOCATED_IN the Permian Region, New Wells Drilled and New Wells Drilled Per Rig are MEASURED_IN the Permian Region, U.S. Energy Information Administration FORECASTS_OUTLOOK for the Permian Region, and Crude Oil and Crude Oil Production From Newly Completed Wells are PRODUCED_IN the Permian Region."
    )


--- RAG Evaluation Report ---
Overall RAG Score: 0.253 / 1.0

Faithfulness Score: 0.3333333333333333
  Reasoning: The context only lists region names without any explicit graph relationships, groupings, or numerical flow data. Only the claim that no explicit graph relationships are defined is supported; the other claims introduce unsupported details about groupings and flows.
  Claims Breakdown:
    - [✓ Supported] Claim: 'There are no explicitly defined graph-based incoming or outgoing structural relationships in the provided context.' | Evidence: 'Appalachia region Bakken region Eagle Ford region Haynesville region Permian region Rest of Lower 48 States'
    - [✗ Unsupported] Claim: 'We can identify relationships based on the groupings and numerical data (positive and negative flows) associated with the Permian region node.' | Evidence: 'None'
    - [✗ Unsupported] Claim: 'For peer group relationships (sibling nodes), the Permian region belongs to the same category/grouping as the fo

In [ ]:
if __name__ == "__main__":    

    evaluator = RAGEvaluator()
    report = evaluator.evaluate(
        question=question,
        context=graph_context,
        answer=graph_answer
    )

    print("--- Graph RAG Evaluation Report ---")
    print(f"Overall Graph RAG Score: {report.composite_score} / 1.0\n")
    print(f"Faithfulness Score: {report.faithfulness.faithfulness_score}")
    print(f"  Reasoning: {report.faithfulness.reasoning}")
    print("  Claims Breakdown:")
    for c in report.faithfulness.claims:
        status = "✓ Supported" if c.is_supported else "✗ Unsupported"
        print(f"    - [{status}] Claim: '{c.claim}' | Evidence: '{c.evidence}'")

    print(f"\nAnswer Relevance Score: {report.answer_relevance.relevance_score}")
    print(f"  Reasoning: {report.answer_relevance.reasoning}")

    print(f"\nContext Precision Score: {report.context_relevance.context_precision_score}")
    print(f"  Reasoning: {report.context_relevance.reasoning}")

--- Graph RAG Evaluation Report ---
Overall Graph RAG Score: 0.972 / 1.0

Faithfulness Score: 1.0
  Reasoning: All extracted claims are directly stated in the provided context sentences, so every claim is supported.
  Claims Breakdown:
    - [✓ Supported] Claim: 'Permian Region is LOCATED_IN United States.' | Evidence: 'The Permian Region is located in the United States.'
    - [✓ Supported] Claim: 'Permian Region is LOCATED_IN Lower 48 States.' | Evidence: 'The Permian Region is located in the Lower 48 States.'
    - [✓ Supported] Claim: 'Permian Region AFFECTS_SUPPLY_OF Total Primary Supply.' | Evidence: 'The Permian Region affects the supply of Total Primary Supply.'
    - [✓ Supported] Claim: 'Permian Region PRODUCES Natural Gas.' | Evidence: 'The Permian Region produces Natural Gas.'
    - [✓ Supported] Claim: 'Cumulative Drilled But Uncompleted Wells is LOCATED_IN the Permian Region.' | Evidence: 'Cumulative Drilled But Uncompleted Wells are located in the Permian Region.'
    - 

Question 2

Which entities contribute to retail pump prices, and how do their structural weightings differ between consuming and producing regions?

In [ ]:
if __name__ == "__main__":
    question = "Which entities contribute to retail pump prices, and how do their structural weightings differ between consuming and producing regions?"
    rag_context = (
        "More facts and figures available in the online Annual Statistical Bulletin. asb.opec.org Source: OPEC ASB and OPEC Secretariat estimates. So the next time you hear that the price of a barrel  of oil  is having an impact on the price you pay at the pump, remember that oil-related taxes are imposed by many governments that are often the biggest beneficiaries.TaxIndustry marginCrude price "
    )
    rag_answer = (
       "Permian region (Series 1 - Activity/Count Metrics): Relationship: Associated with the following data points: '443 458 454 449 458 - - - - - - - 451 - -' Permian formations (Production/Geological Metrics): Relationship: Associated with the following data points: '5.77 5.92 6.04 6.08 5.94 - - - - - - - 5.96 - -' Permian region (Series 2 - Net Change/Flow Metrics): Relationship: Associated with the following negative-value data points: '-437.2 -428.1 -430.2 -432.1 -453.8 - - - - - - - -431.9 - -' Permian region (Series 3 - Regional Production/Volume Metrics): Relationship: Associated with the following 15-period data points: '6.41 6.53 6.70 6.68 6.55 6.62 6.75 6.77 6.87 7.09 7.23 7.34 6.58 6.67 7.14'"
    )

    evaluator = RAGEvaluator()
    report = evaluator.evaluate(
        question=question,
        context=rag_context,
        answer=rag_answer
    )

    print("--- RAG Evaluation Report ---")
    print(f"Overall RAG Score: {report.composite_score} / 1.0\n")
    print(f"Faithfulness Score: {report.faithfulness.faithfulness_score}")
    print(f"  Reasoning: {report.faithfulness.reasoning}")
    print("  Claims Breakdown:")
    for c in report.faithfulness.claims:
        status = "✓ Supported" if c.is_supported else "✗ Unsupported"
        print(f"    - [{status}] Claim: '{c.claim}' | Evidence: '{c.evidence}'")

    print(f"\nAnswer Relevance Score: {report.answer_relevance.relevance_score}")
    print(f"  Reasoning: {report.answer_relevance.reasoning}")

    print(f"\nContext Precision Score: {report.context_relevance.context_precision_score}")
    print(f"  Reasoning: {report.context_relevance.reasoning}")
    

--- RAG Evaluation Report ---
Overall RAG Score: 0.06 / 1.0

Faithfulness Score: 0.0
  Reasoning: The provided context discusses OPEC statistical bulletins, oil taxes, and related topics, but contains no information about the Permian region, series identifiers, or the specific numeric data points listed in the answer. Therefore, none of the extracted claims are supported by the context.
  Claims Breakdown:
    - [✗ Unsupported] Claim: 'Permian region (Series 1 - Activity/Count Metrics): Relationship: Associated with the following data points: '443 458 454 449 458 - - - - - - - 451 - -'' | Evidence: 'None'
    - [✗ Unsupported] Claim: 'Permian formations (Production/Geological Metrics): Relationship: Associated with the following data points: '5.77 5.92 6.04 6.08 5.94 - - - - - - - 5.96 - -'' | Evidence: 'None'
    - [✗ Unsupported] Claim: 'Permian region (Series 2 - Net Change/Flow Metrics): Relationship: Associated with the following negative-value data points: '-437.2 -428.1 -430.2 -

In [ ]:
if __name__ == "__main__":  

    question = "Which entities contribute to retail pump prices, and how do their structural weightings differ between consuming and producing regions?"
    graph_context = (
        "The local subgraph for the Permian Region includes the following entities and relationships: Permian Region is LOCATED_IN the United States and the Lower 48 States. Cumulative Drilled But Uncompleted Wells is LOCATED_IN the Permian Region. New Wells Drilled and New Wells Drilled Per Rig are MEASURED_IN the Permian Region. Permian Region AFFECTS_SUPPLY_OF Total Primary Supply. U.S. Energy Information Administration FORECASTS_OUTLOOK for the Permian Region. Crude Oil and Crude Oil Production From Newly Completed Wells are PRODUCED_IN the Permian Region. Permian Region PRODUCES Natural Gas."
    )
    graph_answer = (
        "Based on the provided data, the pricing benchmarks Brent and Henry Hub are connected to the domestic production basin of the Gulf Of America, while there are no connections associated with global supply chokepoints. Specifically, their relationships in the graph are structured as follows: Brent is linked to the Gulf Of America basin through pathways involving OPEC (which forecasts and affects the supply of crude oil) and the North Sea (where Brent is located and produced), while Henry Hub is connected to the Gulf Of America basin through domestic pathways, specifically via its location in the United States (U.S.), which is linked to the production of crude oil and lease condensate in the Gulf Of America."
    )  
    evaluator = RAGEvaluator()
    report = evaluator.evaluate(
        question=question,
        context=graph_context,
        answer=graph_answer
    )

    print("--- Graph RAG Evaluation Report ---")
    print(f"Overall Graph RAG Score: {report.composite_score} / 1.0\n")
    print(f"Faithfulness Score: {report.faithfulness.faithfulness_score}")
    print(f"  Reasoning: {report.faithfulness.reasoning}")
    print("  Claims Breakdown:")
    for c in report.faithfulness.claims:
        status = "✓ Supported" if c.is_supported else "✗ Unsupported"
        print(f"    - [{status}] Claim: '{c.claim}' | Evidence: '{c.evidence}'")

    print(f"\nAnswer Relevance Score: {report.answer_relevance.relevance_score}")
    print(f"  Reasoning: {report.answer_relevance.reasoning}")

    print(f"\nContext Precision Score: {report.context_relevance.context_precision_score}")
    print(f"  Reasoning: {report.context_relevance.reasoning}")

--- Graph RAG Evaluation Report ---
Overall Graph RAG Score: 0.052 / 1.0

Faithfulness Score: 0.0
  Reasoning: None of the factual statements in the answer are present in the provided context, which only describes entities and relationships related to the Permian Region, United States, and associated metrics. Consequently, all extracted claims are unsupported, resulting in a faithfulness score of 0.0.
  Claims Breakdown:
    - [✗ Unsupported] Claim: 'Brent is connected to the domestic production basin of the Gulf Of America.' | Evidence: 'None'
    - [✗ Unsupported] Claim: 'Henry Hub is connected to the domestic production basin of the Gulf Of America.' | Evidence: 'None'
    - [✗ Unsupported] Claim: 'There are no connections associated with global supply chokepoints.' | Evidence: 'None'
    - [✗ Unsupported] Claim: 'Brent is linked to the Gulf Of America basin through pathways involving OPEC.' | Evidence: 'None'
    - [✗ Unsupported] Claim: 'OPEC forecasts and affects the supply of cr

Question 3

Compare the regulatory and fiscal influence of consuming governments against the market influence of major oil-producing cartels.

In [ ]:
if __name__ == "__main__":
    question = "Compare the regulatory and fiscal influence of consuming governments against the market influence of major oil-producing cartels."
    rag_context = (
        "Comparison of composite barrel (OECD average) An n u a l average OECD retail revenues An n u a l average OPEC oil revenues $tn/year % change in a composite  barrel shares  Estimated annual average revenues (OECD Estimated annual average revenues (OECD Composite barrel components (OECD in Difference $1.972 trillion 2.571 0.599 (OECD retail vs. OPEC oil, 2020–24)"
    )
    rag_answer = (
       "Consuming Governments (OECD): Consuming governments capture the vast majority of the financial value from oil sales. Over the 2020–2024 period, OECD retail revenues from petroleum products averaged **$2.571 trillion per year. Producing Cartels (OPEC): In contrast, OPEC countries generated an average of $0.599 trillion per year from their oil revenues. The Gap:OECD governments earned an average of $1.972 trillion per year more from the retail sale of petroleum products than OPEC countries earned from the original sale of their oil. The physical breakdown of a composite barrel shows that the tax levied by consuming governments far outweighs the price of the crude oil itself: Government Tax Share:113.48 (the largest single component of the composite barrel). Oil Producers (Crude Price): 80.21. Industry Margin:52.73. Taxation Percentage:In 2024, the average tax share on the final retail price in OECD countries reached approximately 46%(a year-on-year increase of 2.8 percentage points). Consequently, the text notes that the real burden on consumers comes from taxes, not from the original price paid for crude oil.Consuming Governments:The trillions of dollars earned from retail oil taxes represent pure income for OECD governments, requiring no direct operational offsets. Producing Cartels:OPEC’s export revenues are not pure profit; they must be used to cover the high capital and operational costs associated with **exploration, production, and transportation of crude oil. While public perception often attributes high fuel costs and massive wealth generation to producing cartels like OPEC, the data shows this is a misconception. Through aggressive fiscal policies and retail taxation, consuming governments (OECD) exert the dominant economic influence, extracting far more revenue from a barrel of oil than the countries that produce it."

    )

    evaluator = RAGEvaluator()
    report = evaluator.evaluate(
        question=question,
        context=rag_context,
        answer=rag_answer
    )

    print("--- RAG Evaluation Report ---")
    print(f"Overall RAG Score: {report.composite_score} / 1.0\n")
    print(f"Faithfulness Score: {report.faithfulness.faithfulness_score}")
    print(f"  Reasoning: {report.faithfulness.reasoning}")
    print("  Claims Breakdown:")
    for c in report.faithfulness.claims:
        status = "✓ Supported" if c.is_supported else "✗ Unsupported"
        print(f"    - [{status}] Claim: '{c.claim}' | Evidence: '{c.evidence}'")

    print(f"\nAnswer Relevance Score: {report.answer_relevance.relevance_score}")
    print(f"  Reasoning: {report.answer_relevance.reasoning}")

    print(f"\nContext Precision Score: {report.context_relevance.context_precision_score}")
    print(f"  Reasoning: {report.context_relevance.reasoning}")


--- RAG Evaluation Report ---
Overall RAG Score: 0.596 / 1.0

Faithfulness Score: 0.54
  Reasoning: The answer contains 13 distinct factual statements. Seven of them are directly supported by the provided context, which includes specific numeric values for OECD retail revenues, OPEC oil revenues, their difference, and the component breakdown of a composite barrel, as well as the 2024 tax share percentage. The remaining six statements are interpretive or introduce information not present in the context, so they are marked as unsupported. The faithfulness score is the ratio of supported claims to total claims (7/13 ≈ 0.54).
  Claims Breakdown:
    - [✗ Unsupported] Claim: 'Consuming governments capture the vast majority of the financial value from oil sales.' | Evidence: 'None'
    - [✓ Supported] Claim: 'Over the 2020–2024 period, OECD retail revenues from petroleum products averaged $2.571 trillion per year.' | Evidence: 'OECD retail revenues ... $2.571 trillion per year.'
    - [✓ Sup

In [8]:
if __name__ == "__main__":    
    
    question = "Compare the regulatory and fiscal influence of consuming governments against the market influence of major oil-producing cartels."
    graph_context = (
        "The Russian Government enacts Export Restrictions, which impacts the price of Commodity Price. The Government enacts Government Subsidies, which impacts the price of Consumer Fuel Prices. Central Bank Of Brazil enacts Monetary Policy, which affects Inflation. Central Bank Of Brazil enacts Monetary Policy, which affects Growth Dynamics. Central Bank Of Brazil enacts Monetary Policy, which affects Interest Rates. Fomc enacts Monetary Policy, which affects Inflation. Fomc enacts Monetary Policy, which affects Growth Dynamics. Fomc enacts Monetary Policy, which affects Interest Rates. Ecb enacts Monetary Policy, which affects Inflation. Ecb enacts Monetary Policy, which affects Growth Dynamics. Across all contexts, Iea produces Iea Subscription Data Services."
    )
    graph_answer = (
        "Based on the regulatory and market dynamics, the influence can be compared as follows: regarding regulatory and fiscal influence of governments and agencies, consuming governments, international bodies, and regulatory agencies influence the oil market through specific policies, reporting standards, and environmental guidance—for example, the Environmental Protection Agency (EPA) regulates crude oil supply through the Epa Flaring Guidance, the Eia (and the U.S. Energy Information Administration) affects the market through reporting and policies like Psa, the Eia-813 Monthly Crude Oil Report, Eia-814, and the Eia-914 Revision Policy, the Iea exerts influence over both crude oil and general oil supply through the Iea Collective Action, and Non-Opec Producers and coalition groups like Opec+ implement policy frameworks such as the Declaration Of Cooperation (Doc) and the Nov 2022 Deal to impact oil supply—whereas regarding the market influence of producing cartels, major oil-producing cartels like Opec exert direct market influence by actively and directly affecting the supply of both Crude Oil and Oil. While regulatory bodies and governments shape the market through compliance, reporting, and collective actions, cartels like OPEC wield direct control over supply levels."
        
    )

    evaluator = RAGEvaluator()
    report = evaluator.evaluate(
        question=question,
        context=graph_context,
        answer=graph_answer
    )

    print("--- Graph RAG Evaluation Report ---")
    print(f"Overall Graph RAG Score: {report.composite_score} / 1.0\n")
    print(f"Faithfulness Score: {report.faithfulness.faithfulness_score}")
    print(f"  Reasoning: {report.faithfulness.reasoning}")
    print("  Claims Breakdown:")
    for c in report.faithfulness.claims:
        status = "✓ Supported" if c.is_supported else "✗ Unsupported"
        print(f"    - [{status}] Claim: '{c.claim}' | Evidence: '{c.evidence}'")

    print(f"\nAnswer Relevance Score: {report.answer_relevance.relevance_score}")
    print(f"  Reasoning: {report.answer_relevance.reasoning}")

    print(f"\nContext Precision Score: {report.context_relevance.context_precision_score}")
    print(f"  Reasoning: {report.context_relevance.reasoning}")

--- Graph RAG Evaluation Report ---
Overall Graph RAG Score: 0.37 / 1.0

Faithfulness Score: 0.0
  Reasoning: None of the extracted claims are mentioned or supported by the provided context, which only references actions of the Russian Government, various central banks, and the IEA's production of subscription data services. Therefore, the answer contains no factual statements grounded in the context.
  Claims Breakdown:
    - [✗ Unsupported] Claim: 'The Environmental Protection Agency (EPA) regulates crude oil supply through the Epa Flaring Guidance.' | Evidence: 'None'
    - [✗ Unsupported] Claim: 'The EIA (U.S. Energy Information Administration) affects the market through reporting and policies like PSA, the EIA-813 Monthly Crude Oil Report, EIA-814, and the EIA-914 Revision Policy.' | Evidence: 'None'
    - [✗ Unsupported] Claim: 'The IEA exerts influence over both crude oil and general oil supply through the IEA Collective Action.' | Evidence: 'None'
    - [✗ Unsupported] Claim: '

Question 4


How would severe storage constraints in Middle Eastern ports alter the causal chain during a transit chokepoint closure?

In [ ]:
if __name__ == "__main__":
    question = "How would severe storage constraints in Middle Eastern ports alter the causal chain during a transit chokepoint closure?"
    rag_context = (
        "Comparison of composite barrel (OECD average) An n u a l average OECD retail revenues An n u a l average OPEC oil revenues $tn/year % change in a composite  barrel shares  Estimated annual average revenues (OECD Estimated annual average revenues (OECD Composite barrel components (OECD in Difference $1.972 trillion 2.571 0.599 (OECD retail vs. OPEC oil, 2020–24)"
    )
    rag_answer = (
       "Based on the provided text, severe storage constraints in Middle Eastern ports alter the causal chain during a transit chokepoint closure (specifically the Strait of Hormuz) in the following ways: first, through forced upstream shut-ins and shutdowns, where when export capacity is lost, inventories quickly build up, and once storage levels reach their maximum limits (critical levels), producers can no longer store excess volume, forcing a physical shutdown of upstream operations. Specifically, for crude oil, producers are forced to shut in additional volumes of crude oil (with shut-ins averaging 11.3 million b/d in May and rising as storage limits, particularly in Iran, are maximized), while for gas and ammonia, upstream gas fields are forced to reduce operations or shut down entirely, and nearly all Middle Eastern ammonia assets are taken offline as stocks reach critical capacity. Second, regarding logistical lag and delayed post-conflict recovery, the massive accumulation of stockpiles alters the recovery timeline, so once the conflict ends and the chokepoint reopens, resuming normal exports is not immediate, and the causal chain is extended because the time required to organize the logistics to deplete these bloated stockpiles is expected to take longer than the actual mechanical process of bringing production facilities back online."

    )

    evaluator = RAGEvaluator()
    report = evaluator.evaluate(
        question=question,
        context=rag_context,
        answer=rag_answer
    )

    print("--- RAG Evaluation Report ---")
    print(f"Overall RAG Score: {report.composite_score} / 1.0\n")
    print(f"Faithfulness Score: {report.faithfulness.faithfulness_score}")
    print(f"  Reasoning: {report.faithfulness.reasoning}")
    print("  Claims Breakdown:")
    for c in report.faithfulness.claims:
        status = "✓ Supported" if c.is_supported else "✗ Unsupported"
        print(f"    - [{status}] Claim: '{c.claim}' | Evidence: '{c.evidence}'")

    print(f"\nAnswer Relevance Score: {report.answer_relevance.relevance_score}")
    print(f"  Reasoning: {report.answer_relevance.reasoning}")

    print(f"\nContext Precision Score: {report.context_relevance.context_precision_score}")
    print(f"  Reasoning: {report.context_relevance.reasoning}")


--- RAG Evaluation Report ---
Overall RAG Score: 0.37 / 1.0

Faithfulness Score: 0.0
  Reasoning: None of the factual claims presented in the answer are found in the provided context, which only contains data about OECD and OPEC revenue comparisons and does not mention storage constraints, the Strait of Hormuz, shut‑ins, or logistical recovery timelines.
  Claims Breakdown:
    - [✗ Unsupported] Claim: 'Severe storage constraints in Middle Eastern ports alter the causal chain during a transit chokepoint closure (Strait of Hormuz) by causing forced upstream shut-ins and shutdowns.' | Evidence: 'None'
    - [✗ Unsupported] Claim: 'When export capacity is lost, inventories quickly build up, and once storage levels reach maximum limits (critical levels), producers can no longer store excess volume, forcing a physical shutdown of upstream operations.' | Evidence: 'None'
    - [✗ Unsupported] Claim: 'For crude oil, producers are forced to shut in additional volumes of crude oil, with shut-in

In [10]:
if __name__ == "__main__":    
    
    question = "How would severe storage constraints in Middle Eastern ports alter the causal chain during a transit chokepoint closure?"
    graph_context = (
        "Fujairah is located in Middle East. Middle East is located in Crude Oil. Crude Oil is transported via Barge. Barge is located in Panama Canal. Crude Oil is transported via Tanker. Tanker is located in Panama Canal. Fujairah is located in Petroleum Product. Petroleum Product affects the supply of Supply. Supply affects the supply of Crude Oil. Crude Oil is located in Strategic Petroleum Reserve. Crude Oil is transported via Railroad. Petroleum Product supplies to Crude Oil. Petroleum Product is transported via Pipelines. Pipelines are transported via Crude Oil. Petroleum Product is transported via Barge. Barge is transported via Crude Oil."

    )
    graph_answer = (
        "Severe storage constraints in Middle Eastern ports, such as Fujairah in the UAE, would alter the causal chain by disrupting the flow and supply of petroleum products and crude oil. Specifically, petroleum products from Fujairah affect the overall supply, are produced in refineries, and are transported via pipelines, barges, and tankers to crude oil in the Strategic Petroleum Reserve. Additionally, crude oil and other products from the Middle East/UAE are transported via barges to transit chokepoints like the Panama Canal. Therefore, storage constraints in these ports would interrupt these refining, transport (via pipelines, barges, and tankers), and supply pathways, ultimately impacting both the Strategic Petroleum Reserve and transit through key chokepoints."
        
    )

    evaluator = RAGEvaluator()
    report = evaluator.evaluate(
        question=question,
        context=graph_context,
        answer=graph_answer
    )

    print("--- Graph RAG Evaluation Report ---")
    print(f"Overall Graph RAG Score: {report.composite_score} / 1.0\n")
    print(f"Faithfulness Score: {report.faithfulness.faithfulness_score}")
    print(f"  Reasoning: {report.faithfulness.reasoning}")
    print("  Claims Breakdown:")
    for c in report.faithfulness.claims:
        status = "✓ Supported" if c.is_supported else "✗ Unsupported"
        print(f"    - [{status}] Claim: '{c.claim}' | Evidence: '{c.evidence}'")

    print(f"\nAnswer Relevance Score: {report.answer_relevance.relevance_score}")
    print(f"  Reasoning: {report.answer_relevance.reasoning}")

    print(f"\nContext Precision Score: {report.context_relevance.context_precision_score}")
    print(f"  Reasoning: {report.context_relevance.reasoning}")

--- Graph RAG Evaluation Report ---
Overall Graph RAG Score: 0.44 / 1.0

Faithfulness Score: 0.42
  Reasoning: Out of 19 extracted factual statements, only 8 are directly supported by the provided context. The unsupported claims introduce causal or location details (e.g., storage constraints effects, refinery production, specific transport destinations) that are not present in the context, resulting in a faithfulness score of 8/19 ≈ 0.42.
  Claims Breakdown:
    - [✓ Supported] Claim: 'Fujairah is a Middle Eastern port.' | Evidence: 'Fujairah is located in Middle East.'
    - [✗ Unsupported] Claim: 'Severe storage constraints in Middle Eastern ports would disrupt the flow and supply of petroleum products.' | Evidence: 'None'
    - [✗ Unsupported] Claim: 'Severe storage constraints in Middle Eastern ports would disrupt the flow and supply of crude oil.' | Evidence: 'None'
    - [✗ Unsupported] Claim: 'Fujairah is in the UAE.' | Evidence: 'None'
    - [✓ Supported] Claim: 'Petroleum prod

Question 5

In Middle East conflict, Strait of Hormuz close make how much crude shut in, and why logistics constraints make it slow to normal?

In [ ]:
if __name__ == "__main__":
    question = "In Middle East conflict, Strait of Hormuz close make how much crude shut in, and why logistics constraints make it slow to normal?"
    rag_context = (
       
        "negotiations. As of this writing, the agreement has not been finalized. Most oil production in the region remains shut in, and global oil inventories have continued to fall to meet demand. Although ships have occasionally transited the strait over the past month, for the purposes of this forecast, we assume that the Strait of Hormuz will remain effectively closed into early summer, with flows slowly starting to resume in the third quarter of 2026 3Q26. If flows resume within this timeframe, we expect it will take until early 2027 for production and trade patterns to generally return to pre conflict status, and we anticipate that some producers around the Persian Gulf will not be able to bring oil output back to pre conflict levels during the STEO forecast period. Disrupted crude oil production volumes in the Middle East increased last month. We assess that production shut ins averaged 11.3 million barrels per day b d in May, and we expect they will continue"
        "the effective closure of the strait of Hormuz and physical damage to oil fields. Similarly, the loss of export capacity has forced upstream gas fields to reduce operations or shut down altogether while nearly all Middle Eastern ammonia assets have been taken offline as stocks reached critical levels. The salient issue here is that for several commodities, resuming exports once the conflict ends will not be straightforward. The time required organise the logistics for depleting stockpiles may, in several instances, be longer than the mechanical processes of bring a facility back online"
        "production shut ins averaged 11.3 million barrels per day b d in May, and we expect they will continue to rise through 2Q26 as storage levels, particularly in Iran, reach maximum limits, requiring producers to shut in additional volumes as the closure of the strait persists. Table 1. Estimated Strait of Hormuz closure related disruptions in crude oil production thousand barrels per day Country Production Feb 26 Estimated shut ins Mar 26 Estimated shut ins Apr 26 Estimated shut ins May 26 Forecast shut ins Jun 26 Forecast shut ins 3Q26 Forecast shut ins 4Q26 Kuwait 2,560 1,400 2,050 1,980 UAE 3,600 1,450 1,100 1,350 Iran 3,390 130 230 780 Iraq 4,400 2,840 3,130 3,190 Qatar 557 450 500 500 Bahrain 193 120 160 160 Saudi Arabia 10,500 2,500 3,200 3,290 Total 25,200 8,890 10,520 11,250 11,340 10,112 5,703 Data source U.S. Energy Information Administration"
        "pilotage at choke points and clearly defined traffic management schemes may be necessary before insurers are willing to return to underwriting risk on standard terms. Following the clearance of any mines, a minimum of two to three months will likely be required to re establish steady export operations, reflecting the need to exit oil laden tankers from the Gulf, reposition ballast tonnage, and rebuild port loading schedules. Upstream production and refining activity are expected to lag the initial resumption of shipping. Countries with port bottlenecks and minimal storage, such as Iraq, will take longer to return to pre war shipping levels. Countries with less congested ports and more resilient supply chains, notably Saudi Arabia and the UAE, are likely to see a quicker return of output, supported by mobilised workforces, efficient port operations, active construction yards and deeper supply chain inventories. Both benefit from"

    )
    rag_answer = (
       "Based on the provided text, severe storage constraints in Middle Eastern ports alter the causal chain during a transit chokepoint closure (specifically the Strait of Hormuz) in the following ways: first, through forced upstream shut-ins and shutdowns, where when export capacity is lost, inventories quickly build up, and once storage levels reach their maximum limits (critical levels), producers can no longer store excess volume, forcing a physical shutdown of upstream operations. Specifically, for crude oil, producers are forced to shut in additional volumes of crude oil (with shut-ins averaging 11.3 million b/d in May and rising as storage limits, particularly in Iran, are maximized), while for gas and ammonia, upstream gas fields are forced to reduce operations or shut down entirely, and nearly all Middle Eastern ammonia assets are taken offline as stocks reach critical capacity. Second, regarding logistical lag and delayed post-conflict recovery, the massive accumulation of stockpiles alters the recovery timeline, so once the conflict ends and the chokepoint reopens, resuming normal exports is not immediate, and the causal chain is extended because the time required to organize the logistics to deplete these bloated stockpiles is expected to take longer than the actual mechanical process of bringing production facilities back online."

    )

    evaluator = RAGEvaluator()
    report = evaluator.evaluate(
        question=question,
        context=rag_context,
        answer=rag_answer
    )

    print("--- RAG Evaluation Report ---")
    print(f"Overall RAG Score: {report.composite_score} / 1.0\n")
    print(f"Faithfulness Score: {report.faithfulness.faithfulness_score}")
    print(f"  Reasoning: {report.faithfulness.reasoning}")
    print("  Claims Breakdown:")
    for c in report.faithfulness.claims:
        status = "✓ Supported" if c.is_supported else "✗ Unsupported"
        print(f"    - [{status}] Claim: '{c.claim}' | Evidence: '{c.evidence}'")

    print(f"\nAnswer Relevance Score: {report.answer_relevance.relevance_score}")
    print(f"  Reasoning: {report.answer_relevance.reasoning}")

    print(f"\nContext Precision Score: {report.context_relevance.context_precision_score}")
    print(f"  Reasoning: {report.context_relevance.reasoning}")


--- RAG Evaluation Report ---
Overall RAG Score: 0.97 / 1.0

Faithfulness Score: 1.0
  Reasoning: All nine extracted statements are directly supported by sentences in the provided context, with explicit evidence for each claim. Therefore, the answer is fully faithful to the source material.
  Claims Breakdown:
    - [✓ Supported] Claim: 'Severe storage constraints in Middle Eastern ports alter the causal chain during a transit chokepoint closure (Strait of Hormuz).' | Evidence: 'as storage levels, particularly in Iran, reach maximum limits, requiring producers to shut in additional volumes as the closure of the strait persists.'
    - [✓ Supported] Claim: 'The alteration occurs first through forced upstream shut-ins and shutdowns when export capacity is lost, inventories quickly build up, and once storage levels reach maximum limits, producers can no longer store excess volume, forcing a physical shutdown of upstream operations.' | Evidence: 'production shut ins averaged 11.3 million b

In [12]:
if __name__ == "__main__":    
    
    question = "In Middle East conflict, Strait of Hormuz close make how much crude shut in, and why logistics constraints make it slow to normal?"
    graph_context = (
        "The closure of the Strait of Hormuz, located in the Strait of Hormuz, affects Gulf countries, the Atlantic Basin, and Asia, while impacting the supply of crude oil, jet fuel, naphtha, oil, LPG/ethane, and diesel."

    )
    graph_answer = (
        "I do not have the information to answer how much crude would be shut in or why logistics constraints make recovery slow. The provided information only indicates that a closure of the Strait of Hormuz affects the supply of crude oil, oil, diesel, jet fuel, naphtha, and LPG/ethane, and impacts Asia, the Atlantic Basin, and Gulf Countries."
        
    )

    evaluator = RAGEvaluator()
    report = evaluator.evaluate(
        question=question,
        context=graph_context,
        answer=graph_answer
    )

    print("--- Graph RAG Evaluation Report ---")
    print(f"Overall Graph RAG Score: {report.composite_score} / 1.0\n")
    print(f"Faithfulness Score: {report.faithfulness.faithfulness_score}")
    print(f"  Reasoning: {report.faithfulness.reasoning}")
    print("  Claims Breakdown:")
    for c in report.faithfulness.claims:
        status = "✓ Supported" if c.is_supported else "✗ Unsupported"
        print(f"    - [{status}] Claim: '{c.claim}' | Evidence: '{c.evidence}'")

    print(f"\nAnswer Relevance Score: {report.answer_relevance.relevance_score}")
    print(f"  Reasoning: {report.answer_relevance.reasoning}")

    print(f"\nContext Precision Score: {report.context_relevance.context_precision_score}")
    print(f"  Reasoning: {report.context_relevance.reasoning}")

--- Graph RAG Evaluation Report ---
Overall Graph RAG Score: 0.28 / 1.0

Faithfulness Score: 0.5
  Reasoning: Four factual statements were extracted from the answer. Two statements about lacking information are not supported by the provided context, while the statements describing the affected supplies and regions are directly supported by the context sentence. Thus, 2 out of 4 claims are supported, yielding a faithfulness score of 0.5.
  Claims Breakdown:
    - [✗ Unsupported] Claim: 'I do not have the information to answer how much crude would be shut in.' | Evidence: 'None'
    - [✗ Unsupported] Claim: 'I do not have the information to answer why logistics constraints make recovery slow.' | Evidence: 'None'
    - [✓ Supported] Claim: 'A closure of the Strait of Hormuz affects the supply of crude oil, oil, diesel, jet fuel, naphtha, and LPG/ethane.' | Evidence: 'The closure of the Strait of Hormuz, located in the Strait of Hormuz, affects Gulf countries, the Atlantic Basin, and Asia,

Question 6

Given the intense market tightness and shipping disruptions caused by the ongoing Middle East conflict, what are the specific conflict implications regarding the loss of global production capacity and the logistical challenges of resuming exports, and how can market participants utilize Argus analytics services to obtain near-term and long-term price forecasts to navigate this volatility?

In [ ]:
if __name__ == "__main__":
    question = "Given the intense market tightness and shipping disruptions caused by the ongoing Middle East conflict, what are the specific conflict implications regarding the loss of global production capacity and the logistical challenges of resuming exports, and how can market participants utilize Argus analytics services to obtain near-term and long-term price forecasts to navigate this volatility?"
    rag_context = (
       
        "2026 Argus Media group. All rights reserved. Foreword Argus Media Analytics Services 2 8 weeks on from the start of the conflict in the Middle East, global commodity markets remain in tumult. As modern societys propensity for immediate, if not enduring, news leaves commodity producers, traders and consumers chasing shadows as they seek to understand when normal operations will resume the seriousness of the long term implications of this conflict are becoming clear, if not always appreciated. At the time of writing, a fragile two week ceasefire agreed between Washington and Tehran remains in place. Despite this détente, the game of chess for control of the strait of Hormuz continues with global commodity shipping very much the pawns being sacrificed. The longer the current situation persists the greater the enduring shortage of the key commodities across global markets will be. Already, physical crude and"
        "2026 Argus Media group. All rights reserved. The implications of the conflict on production capacity 3 Argus Media Analytics Services 0 10 20 30 40 50 60 70 80 90 100 Sulphur Ammonia Methanol Crude LNG Ethylene Urea Refined Products LPG Capacity lost Unimpacted RoW Unimpacted ME Impacted ME The conflict has, inevitably, had major direct and indirect impact on the production infrastructure of many commodities, removing in excess of 15pc of global capacity for some. But, while numerous energy linked facilities and installations across Mideast Gulf countries have been damaged a meaningful proportion of lost capacity has also stemmed from precautionary closures or logistics constraints. For example, more than 8mn b d of crude exports were shut in in March owing to both the effective closure of the strait of Hormuz and physical damage to oil fields. Similarly, the loss of export capacity has forced upstream gas fields to reduce operations or"
        "2026 Argus Media group. All rights reserved. Forecasted scenario assuming the strait opens on 1 MaySulphur Ammonia Methanol Crude LNG Urea Refined Products LPG 20 40 60 80 100 6 mth av. Mar 26 Apr 26 May 26 Jun 26 Jul 26 Aug 26 Sep 26 Oct 26 Nov 26 Dec 26 The level of disruption to export supply Average global production from Sept 25 to Feb 26 100 The implications of the conflict on supply 4 Argus Media Analytics Services The strait of Hormuz is the vital tributary for exports from the region and so, with vessel traffic reduced to just 5pc of typical flows in March, the consequences were dramatic. Interestingly, this has been the case for markets with tight and surplus balances. For both LPG and sulphur, this shock hit previously oversupplied markets, triggering uncertainty as key importers scrambled for alternative cargoes, tightening near term availability and increasing reliance on inventories. Additional stress on the broader sulphur market has"

    )
    rag_answer = (
       "Based on the provided context, here are the specific implications and logistical challenges, as well as the details regarding Argus analytics services 1. Implications for Global Production Capacity Significant Capacity Loss The conflict has removed in excess of 15 of global capacity for certain commodities with impacts felt across sulphur, ammonia, methanol, crude, LNG, ethylene, urea, refined products, and LPG. Direct Damage vs. Precautionary Closures While numerous energy linked facilities and installations across Mideast Gulf countries have suffered physical damage, a meaningful share of lost capacity has stemmed from precautionary shutdowns and logistical bottlenecks. Crude and Gas Shut ins More than 8mn b d of crude exports were shut in in March due to physical damage to oil fields combined with the effective closure of the Strait of Hormuz. Similarly, lost export capacity has forced upstream gas fields to reduce operations or shut in production. 2. Logistical Challenges of Resuming Exports Strait of Hormuz Bottleneck The Strait of Hormuz serves as the vital export artery, but vessel traffic plunged to just 5 of typical flows in March. Geopolitical Vulnerability for Shipping Despite a fragile two week ceasefire between Washington and Tehran, the contest for control of the strait continues, leaving commercial commodity shipping caught in the middle and vulnerable. Supply Scramble and Inventory Depletion Even previously oversupplied markets such as LPG and sulphur experienced severe disruptions, forcing key importers to scramble for alternative cargoes, increasing dependence on inventories, and tightening near term availability. How to Utilize Argus Analytics Services for Price Forecasts The provided text notes that Argus Media Analytics Services led by Simon Morris, Head of Analytics provides insights addressing how much capacity has been taken offline, total supply lost, potential supply ramp up trajectories, and the broader implications for global commodity markets. However, the provided context does not mention specific instructions or methods for how market participants can utilize Argus services to obtain near term and long term price forecasts."

    )

    evaluator = RAGEvaluator()
    report = evaluator.evaluate(
        question=question,
        context=rag_context,
        answer=rag_answer
    )

    print("--- RAG Evaluation Report ---")
    print(f"Overall RAG Score: {report.composite_score} / 1.0\n")
    print(f"Faithfulness Score: {report.faithfulness.faithfulness_score}")
    print(f"  Reasoning: {report.faithfulness.reasoning}")
    print("  Claims Breakdown:")
    for c in report.faithfulness.claims:
        status = "✓ Supported" if c.is_supported else "✗ Unsupported"
        print(f"    - [{status}] Claim: '{c.claim}' | Evidence: '{c.evidence}'")

    print(f"\nAnswer Relevance Score: {report.answer_relevance.relevance_score}")
    print(f"  Reasoning: {report.answer_relevance.reasoning}")

    print(f"\nContext Precision Score: {report.context_relevance.context_precision_score}")
    print(f"  Reasoning: {report.context_relevance.reasoning}")


--- RAG Evaluation Report ---
Overall RAG Score: 0.776 / 1.0

Faithfulness Score: 0.8461538461538461
  Reasoning: Out of 13 factual statements extracted from the answer, 11 are directly supported by sentences in the supplied context. Two statements about Argus leadership and the specific content of Argus services are not mentioned in the context, making them unsupported. The faithfulness score is the ratio of supported claims to total claims (11/13 ≈ 0.846).
  Claims Breakdown:
    - [✓ Supported] Claim: 'The conflict has removed in excess of 15% of global capacity for certain commodities.' | Evidence: 'the conflict has, inevitably, had major direct and indirect impact on the production infrastructure of many commodities, removing in excess of 15pc of global capacity for some.'
    - [✓ Supported] Claim: 'The impacted commodities include sulphur, ammonia, methanol, crude, LNG, ethylene, urea, refined products, and LPG.' | Evidence: 'Sulphur Ammonia Methanol Crude LNG Ethylene Urea Refi

In [14]:
if __name__ == "__main__":    
    
    question = "Given the intense market tightness and shipping disruptions caused by the ongoing Middle East conflict, what are the specific conflict implications regarding the loss of global production capacity and the logistical challenges of resuming exports, and how can market participants utilize Argus analytics services to obtain near-term and long-term price forecasts to navigate this volatility?"
    graph_context = (
    "The conflict implications of the Middle East conflict have resulted in the removal of over 15% of global production capacity for certain commodities. This loss stems from physical damage to energy-linked facilities, precautionary closures, and severe logistics constraints, such as the shut-in of over 8 million barrels per day of crude exports and the halting of nearly all Middle Eastern ammonia assets. Resuming exports will be highly complex; organizing the logistics to deplete critical stockpiles may take longer than mechanically bringing facilities back online, and ongoing threats to regional energy infrastructure raise doubts about whether countries can ever return to pre-war production levels. To navigate these disruptions, market participants can use Argus's analytics services to secure reliable price forecasts. Argus provides monthly Near-term outlooks featuring 12-24 month forecasts for prices, supply, demand, and trade, as well as Long-term analytics that offer comprehensive analysis of future pricing, costs, and capacity looking as far as 25 years ahead."

    )
    graph_answer = (
        "Disruption to Middle East flows impacts the market by directly affecting spot crude oil prices and Dubai-linked benchmarks, as well as impacting the price of North Sea Dated. To help market participants navigate this volatility, Argus Media and Argus Analytics monitor these disruptions, with Argus Media providing outlook forecasts for key products such as Fuel Oil and Naphtha."
        
    )

    evaluator = RAGEvaluator()
    report = evaluator.evaluate(
        question=question,
        context=graph_context,
        answer=graph_answer
    )

    print("--- Graph RAG Evaluation Report ---")
    print(f"Overall Graph RAG Score: {report.composite_score} / 1.0\n")
    print(f"Faithfulness Score: {report.faithfulness.faithfulness_score}")
    print(f"  Reasoning: {report.faithfulness.reasoning}")
    print("  Claims Breakdown:")
    for c in report.faithfulness.claims:
        status = "✓ Supported" if c.is_supported else "✗ Unsupported"
        print(f"    - [{status}] Claim: '{c.claim}' | Evidence: '{c.evidence}'")

    print(f"\nAnswer Relevance Score: {report.answer_relevance.relevance_score}")
    print(f"  Reasoning: {report.answer_relevance.reasoning}")

    print(f"\nContext Precision Score: {report.context_relevance.context_precision_score}")
    print(f"  Reasoning: {report.context_relevance.reasoning}")

--- Graph RAG Evaluation Report ---
Overall Graph RAG Score: 0.282 / 1.0

Faithfulness Score: 0.0
  Reasoning: None of the extracted claims are mentioned or implied in the provided context. The context discusses general production capacity loss, logistics constraints, and Argus's analytics services (monthly near‑term outlooks and long‑term analysis) without referencing spot crude prices, Dubai benchmarks, North Sea Dated, monitoring activities, or specific product forecasts for Fuel Oil or Naphtha. Therefore, all claims are unsupported, resulting in a faithfulness score of 0.0.
  Claims Breakdown:
    - [✗ Unsupported] Claim: 'Disruption to Middle East flows impacts the market by directly affecting spot crude oil prices.' | Evidence: 'None'
    - [✗ Unsupported] Claim: 'Disruption to Middle East flows impacts the market by directly affecting Dubai-linked benchmarks.' | Evidence: 'None'
    - [✗ Unsupported] Claim: 'Disruption to Middle East flows impacts the market by impacting the pri

Question 7

Middle East conflict Strait of Hormuz shipping disruptions impact and Argus price forecasts?

In [ ]:
if __name__ == "__main__":
    question = "Middle East conflict Strait of Hormuz shipping disruptions impact and Argus price forecasts?"
    rag_context = (
        "2026 Argus Media group. All rights reserved. Foreword Argus Media Analytics Services 2 8 weeks on from the start of the conflict in the Middle East, global commodity markets remain in tumult. As modern societys propensity for immediate, if not enduring, news leaves commodity producers, traders and consumers chasing shadows as they seek to understand when normal operations will resume the seriousness of the long term implications of this conflict are becoming clear, if not always appreciated. At the time of writing, a fragile two week ceasefire agreed between Washington and Tehran remains in place. Despite this détente, the game of chess for control of the strait of Hormuz continues with global commodity shipping very much the pawns being sacrificed. The longer the current situation persists the greater the enduring shortage of the key commodities across global markets will be. Already, physical crude and"
        "2026 Argus Media group. All rights reserved. Forecasted scenario assuming the strait opens on 1 MaySulphur Ammonia Methanol Crude LNG Urea Refined Products LPG 20 40 60 80 100 6 mth av. Mar 26 Apr 26 May 26 Jun 26 Jul 26 Aug 26 Sep 26 Oct 26 Nov 26 Dec 26 The level of disruption to export supply Average global production from Sept 25 to Feb 26 100 The implications of the conflict on supply 4 Argus Media Analytics Services The strait of Hormuz is the vital tributary for exports from the region and so, with vessel traffic reduced to just 5pc of typical flows in March, the consequences were dramatic. Interestingly, this has been the case for markets with tight and surplus balances. For both LPG and sulphur, this shock hit previously oversupplied markets, triggering uncertainty as key importers scrambled for alternative cargoes, tightening near term availability and increasing reliance on inventories. Additional stress on the broader sulphur market has"
        "negotiations. As of this writing, the agreement has not been finalized. Most oil production in the region remains shut in, and global oil inventories have continued to fall to meet demand. Although ships have occasionally transited the strait over the past month, for the purposes of this forecast, we assume that the Strait of Hormuz will remain effectively closed into early summer, with flows slowly starting to resume in the third quarter of 2026 3Q26. If flows resume within this timeframe, we expect it will take until early 2027 for production and trade patterns to generally return to pre conflict status, and we anticipate that some producers around the Persian Gulf will not be able to bring oil output back to pre conflict levels during the STEO forecast period. Disrupted crude oil production volumes in the Middle East increased last month. We assess that production shut ins averaged 11.3 million barrels per day b d in May, and we expect they will continue"
        "blockade, initially focused on ships heading to and from Iran s ports, later expanded to shadow fleet vessels carrying Iran s oil exports. Dated fell briefly below 100 bbl on 17 April after President Trump announced that a deal with Iran was all but agreed and the Strait fully open. However, prices rebounded after Iran refuted this, resulting in a prolonged stalemate amid like for like vessel seizures, with shipments through the Strait remaining heavily curtailed. President Trump extended the truce just before its expiry, but further planned diplomatic talks in Pakistan were delayed indefinitely. However, oil prices fell by 10 bbl on 6 May on reports that the United States and Iran were close to agreeing a deal to end the war and reopen the Strait. Five days later, the United States rejected Iran s terms, causing prices to rebound. At the time of writing, Hormuz remained closed and Dated was trading around 110 bbl, up more than 50 from pre war levels"

    )
    rag_answer = (
        "Based on the provided context, the Middle East conflict and shipping disruptions in the Strait of Hormuz have had the following impacts on global commodity markets and oil prices 1. Shipping Disruption and Upstream Impact Drastic Shipping Reductions Vessel traffic through the Strait of Hormuz fell to just 5 of typical flows in March. Shipments remain heavily curtailed by blockades, shadow fleet targeting, and like for like vessel seizures. Massive Production Shut ins Middle East crude oil production shut ins averaged 11.3 million barrels per day b d in May, with most regional oil production remaining shut in. Broad Commodity Disruption Disrupted exports span physical crude, refined products, LNG, LPG, sulphur, ammonia, urea, and methanol. Even previously oversupplied markets such as LPG and sulphur experienced acute supply shocks, forcing key importers to rely heavily on falling inventories and scramble for replacement cargoes. Prolonged Recovery Timeline Forecasts assume the Strait remains effectively closed into early summer, with flows slowly resuming in the third quarter of 2026 3Q26. A general return to pre conflict production and trade patterns is not expected until early 2027, with some Persian Gulf producers expected to remain below pre conflict output levels throughout the forecast period. 2. Crude Price Movements and Current Levels Price Volatility Oil prices have swung sharply around diplomatic headlines 17 April Dated Brent briefly fell below 100 bbl following announcements of a potential deal, rebounding shortly after when the agreement was refuted. 6 May Prices dropped by 10 bbl on reports that a deal was imminent, but rebounded five days later after the U.S. rejected Iran s terms. Prevailing Price Level At the time of writing, with the Strait remaining closed, Dated Brent was trading around 110 bbl, marking an increase of more than 50 above pre war levels."

    )

    evaluator = RAGEvaluator()
    report = evaluator.evaluate(
        question=question,
        context=rag_context,
        answer=rag_answer
    )

    print("--- RAG Evaluation Report ---")
    print(f"Overall RAG Score: {report.composite_score} / 1.0\n")
    print(f"Faithfulness Score: {report.faithfulness.faithfulness_score}")
    print(f"  Reasoning: {report.faithfulness.reasoning}")
    print("  Claims Breakdown:")
    for c in report.faithfulness.claims:
        status = "✓ Supported" if c.is_supported else "✗ Unsupported"
        print(f"    - [{status}] Claim: '{c.claim}' | Evidence: '{c.evidence}'")

    print(f"\nAnswer Relevance Score: {report.answer_relevance.relevance_score}")
    print(f"  Reasoning: {report.answer_relevance.reasoning}")

    print(f"\nContext Precision Score: {report.context_relevance.context_precision_score}")
    print(f"  Reasoning: {report.context_relevance.reasoning}")


--- RAG Evaluation Report ---
Overall RAG Score: 0.824 / 1.0

Faithfulness Score: 1.0
  Reasoning: All ten extracted factual statements are directly corroborated by sentences in the provided context, giving a perfect match between the answer and the source material.
  Claims Breakdown:
    - [✓ Supported] Claim: 'Vessel traffic through the Strait of Hormuz fell to just 5% of typical flows in March.' | Evidence: 'vessel traffic reduced to just 5pc of typical flows in March'
    - [✓ Supported] Claim: 'Shipments remain heavily curtailed by blockades, shadow fleet targeting, and like‑for‑like vessel seizures.' | Evidence: 'blockade, initially focused on ships heading to and from Iran's ports, later expanded to shadow fleet vessels carrying Iran's oil exports. ... like for like vessel seizures'
    - [✓ Supported] Claim: 'Middle East crude oil production shut‑ins averaged 11.3 million barrels per day in May, with most regional oil production remaining shut in.' | Evidence: 'production shut

In [16]:
if __name__ == "__main__":    
    
    question = "Middle East conflict Strait of Hormuz shipping disruptions impact and Argus price forecasts?"
    graph_context = (
    "The Middle East conflict has caused severe disruptions to global commodity markets, reducing vessel traffic through the Strait of Hormuz to just 5% of typical flows. This has led to intense market tightness, with jet fuel prices doubling pre-war levels and diesel cracks reaching $50/bl. Even if the strait reliably reopens, there will be a considerable lag before markets normalize, and volatile commodities like ammonia will be the last to risk shipping. To help navigate these ongoing disruptions, Argus provides near-term outlooks featuring 12-24 month price forecasts for supply, demand, and trade, alongside long-term analytics that project up to 25 years ahead."

    )
    graph_answer = (
        "Disruption to Middle East flows affects Dubai-Linked Benchmarks and Spot Crude Oil Prices, and impacts the price of North Sea Dated. Additionally, Argus Media forecasts the outlook for Fuel Oil, Naphtha, Gasoline, and Jet/Kerosene."
        
    )

    evaluator = RAGEvaluator()
    report = evaluator.evaluate(
        question=question,
        context=graph_context,
        answer=graph_answer
    )

    print("--- Graph RAG Evaluation Report ---")
    print(f"Overall Graph RAG Score: {report.composite_score} / 1.0\n")
    print(f"Faithfulness Score: {report.faithfulness.faithfulness_score}")
    print(f"  Reasoning: {report.faithfulness.reasoning}")
    print("  Claims Breakdown:")
    for c in report.faithfulness.claims:
        status = "✓ Supported" if c.is_supported else "✗ Unsupported"
        print(f"    - [{status}] Claim: '{c.claim}' | Evidence: '{c.evidence}'")

    print(f"\nAnswer Relevance Score: {report.answer_relevance.relevance_score}")
    print(f"  Reasoning: {report.answer_relevance.reasoning}")

    print(f"\nContext Precision Score: {report.context_relevance.context_precision_score}")
    print(f"  Reasoning: {report.context_relevance.reasoning}")

--- Graph RAG Evaluation Report ---
Overall Graph RAG Score: 0.52 / 1.0

Faithfulness Score: 0.0
  Reasoning: None of the extracted claims are directly mentioned or supported by the provided context. The context discusses general market disruptions, jet fuel price spikes, diesel cracks, and Argus's provision of near‑term and long‑term outlooks, but it does not reference Dubai‑Linked benchmarks, spot crude prices, North Sea Dated, or specific product forecasts for fuel oil, naphtha, gasoline, or jet/kerosene.
  Claims Breakdown:
    - [✗ Unsupported] Claim: 'Disruption to Middle East flows affects Dubai-Linked Benchmarks.' | Evidence: 'None'
    - [✗ Unsupported] Claim: 'Disruption to Middle East flows affects Spot Crude Oil Prices.' | Evidence: 'None'
    - [✗ Unsupported] Claim: 'Disruption to Middle East flows impacts the price of North Sea Dated.' | Evidence: 'None'
    - [✗ Unsupported] Claim: 'Argus Media forecasts the outlook for Fuel Oil.' | Evidence: 'None'
    - [✗ Unsupported

Question 8

What key geopolitical conflict caused Middle East crude oil producers to cut production by more than 11 million barrels per day in May 2026?

In [ ]:
if __name__ == "__main__":
    question = "What key geopolitical conflict caused Middle East crude oil producers to cut production by more than 11 million barrels per day in May 2026?"
    rag_context = (
        "negotiations. As of this writing, the agreement has not been finalized. Most oil production in the region \nremains shut-in, and global oil inventories have continued to fall to meet demand. \nAlthough ships have occasionally transited the strait over the past month, for the purposes of this \nforecast, we assume that the Strait of Hormuz will remain effectively closed into early summer, with \nflows slowly starting to resume in the third quarter of 2026 (3Q26). If flows resume within this \ntimeframe, we expect it will take until early 2027 for production and trade patterns to generally return \nto pre-conflict status, and we anticipate that some producers around the Persian Gulf will not be able to \nbring oil output back to pre-conflict levels during the STEO forecast period. \nDisrupted crude oil production volumes in the Middle East increased last month. We assess that \nproduction shut-ins averaged 11.3 million barrels per day (b/d) in May, and we expect they will continue')"
        "13 May 2026 \n• World oil demand is forecast to contract by 420 kb/d y -o-y in 2026, to 104 mb/d, 1.3 mb/d \nless than our pre-war forecast. The biggest decline is in 2Q26, down by 2.45 mb/d, of which \nthe OECD accounts for 930 kb/d and the non- OECD for 1.5 mb/d. The petrochemical and \naviation sectors are currently most affected, but higher prices, a weaker economic \nenvironment and demand-saving measures will increasingly impact fuel use. \n• Global oil supply declined by a further 1.8 mb/d in April to 95.1 mb/d, taking total losses \nsince February to 12.8 mb/d. Output from Gulf countries affected by the closure of the Strait \nof Hormuz was 14.4 mb/d below pre- war levels. Higher production and ex ports from the \nAtlantic Basin provide some relief. Assuming flows through the Strait gradually resume from \nJune, global oil supply is projected to decline by 3.9 mb/d on average in 2026, to 102.2 mb/d.')"
        "Global oil inventories. Global oil markets remain highly volatile as very limited shipping traffic \nthrough the Strait of Hormuz has caused oil producers in the Middle East to reduce crude oil \nproduction by more than 11 million barrels per day (b/d) in May compared with pre-conflict \nlevels. This drop in production has resulted in large global inventory draws to meet demand. \nUnder our assumptions, we expect global oil inventories will fall by an average of 6.3 million b/d \nin 2Q26 and by 7.6 million b/d in 3Q26. Oil inventories in the Organization for Economic \nCooperation and Development in our forecast fall to their lowest levels since 2003. \n• Global oil consumption. High fuel prices, reduced fuel availability, and government initiatives \nhave lowered oil demand. As a result, we now forecast that global oil demand will decrease by \n1.1 million b/d over the course of 2026, compared to 104.0 in 2025. In our May STEO we \nShort-Term Energy Outlook')"
        "production shut-ins averaged 11.3 million barrels per day (b/d) in May, and we expect they will continue \nto rise through 2Q26 as storage levels, particularly in Iran, reach maximum limits, requiring producers to \nshut in additional volumes as the closure of the strait persists. \nTable 1. Estimated Strait of Hormuz closure-related disruptions in crude oil production  \nthousand barrels per day \nCountry \nProduction  \nFeb-26 \nEstimated  \nshut-ins \nMar-26 \nEstimated  \nshut-ins \nApr-26 \nEstimated \nshut-ins \nMay-26 \nForecast  \nshut-ins \nJun-26 \nForecast  \nshut-ins \n3Q26 \nForecast  \nshut-ins \n4Q26 \nKuwait 2,560 1,400 2,050 1,980    \nUAE 3,600 1,450 1,100 1,350    \nIran 3,390 130 230 780    \nIraq 4,400 2,840 3,130 3,190    \nQatar 557 450 500 500    \nBahrain 193 120 160 160    \nSaudi Arabia 10,500 2,500 3,200 3,290    \nTotal 25,200 8,890 10,520 11,250 11,340 10,112 5,703 \nData source: U.S. Energy Information Administration')]"

    )
    rag_answer = (
        "Based on the provided context, the text attributes the production reduction of more than 11 million barrels per day in May 2026 to the **effective closure of the Strait of Hormuz** and severely restricted shipping traffic through the strait. \n\nHowever, the context does not name or provide specific details about the underlying **geopolitical conflict** itself, referring to it only generically as a conflict or war."

    )

    evaluator = RAGEvaluator()
    report = evaluator.evaluate(
        question=question,
        context=rag_context,
        answer=rag_answer
    )

    print("--- RAG Evaluation Report ---")
    print(f"Overall RAG Score: {report.composite_score} / 1.0\n")
    print(f"Faithfulness Score: {report.faithfulness.faithfulness_score}")
    print(f"  Reasoning: {report.faithfulness.reasoning}")
    print("  Claims Breakdown:")
    for c in report.faithfulness.claims:
        status = "✓ Supported" if c.is_supported else "✗ Unsupported"
        print(f"    - [{status}] Claim: '{c.claim}' | Evidence: '{c.evidence}'")

    print(f"\nAnswer Relevance Score: {report.answer_relevance.relevance_score}")
    print(f"  Reasoning: {report.answer_relevance.reasoning}")

    print(f"\nContext Precision Score: {report.context_relevance.context_precision_score}")
    print(f"  Reasoning: {report.context_relevance.reasoning}")


--- RAG Evaluation Report ---
Overall RAG Score: 0.7 / 1.0

Faithfulness Score: 1.0
  Reasoning: Both extracted claims are directly supported by statements in the provided context. The first claim is backed by a sentence linking limited Strait of Hormuz traffic to a >11 million b/d production drop in May. The second claim is supported by the context's use of generic terms like 'pre-conflict' and 'pre-war' without specifying a named conflict, confirming that no specific geopolitical details are given.
  Claims Breakdown:
    - [✓ Supported] Claim: 'The production reduction of more than 11 million barrels per day in May 2026 is attributed to the effective closure of the Strait of Hormuz and severely restricted shipping traffic through the strait.' | Evidence: 'very limited shipping traffic through the Strait of Hormuz has caused oil producers in the Middle East to reduce crude oil production by more than 11 million barrels per day (b/d) in May compared with pre-conflict levels.'
    - [✓

In [18]:
if __name__ == "__main__":    
    
    question = "What key geopolitical conflict caused Middle East crude oil producers to cut production by more than 11 million barrels per day in May 2026?"
    graph_context = (
    
    "These items represent key themes surrounding geopolitical conflict and warfare in the Middle East, along with their impact on energy supply flows and refinery operations."

    )
    graph_answer = (
        "The key geopolitical conflict associated with the production cuts is the Middle East Conflict (also referred to as the War In The Middle East or Middle East War), alongside Middle East Geopolitical Tensions, Attacks On Refineries, and Disruption To Middle East Flows."
        
    )

    evaluator = RAGEvaluator()
    report = evaluator.evaluate(
        question=question,
        context=graph_context,
        answer=graph_answer
    )

    print("--- Graph RAG Evaluation Report ---")
    print(f"Overall Graph RAG Score: {report.composite_score} / 1.0\n")
    print(f"Faithfulness Score: {report.faithfulness.faithfulness_score}")
    print(f"  Reasoning: {report.faithfulness.reasoning}")
    print("  Claims Breakdown:")
    for c in report.faithfulness.claims:
        status = "✓ Supported" if c.is_supported else "✗ Unsupported"
        print(f"    - [{status}] Claim: '{c.claim}' | Evidence: '{c.evidence}'")

    print(f"\nAnswer Relevance Score: {report.answer_relevance.relevance_score}")
    print(f"  Reasoning: {report.answer_relevance.reasoning}")

    print(f"\nContext Precision Score: {report.context_relevance.context_precision_score}")
    print(f"  Reasoning: {report.context_relevance.reasoning}")

--- Graph RAG Evaluation Report ---
Overall Graph RAG Score: 0.24 / 1.0

Faithfulness Score: 0.0
  Reasoning: The provided context only mentions that there are key themes related to geopolitical conflict and warfare in the Middle East and their impact on energy flows and refinery operations, without specifying any of the exact terms or linking them to production cuts. Therefore, none of the detailed claims in the answer are directly supported by the context.
  Claims Breakdown:
    - [✗ Unsupported] Claim: 'The key geopolitical conflict associated with the production cuts is the Middle East Conflict.' | Evidence: 'None'
    - [✗ Unsupported] Claim: 'The Middle East Conflict is also referred to as the War In The Middle East.' | Evidence: 'None'
    - [✗ Unsupported] Claim: 'The Middle East Conflict is also referred to as Middle East War.' | Evidence: 'None'
    - [✗ Unsupported] Claim: 'Middle East Geopolitical Tensions are a key theme alongside the conflict.' | Evidence: 'None'
    - [

Question 9

Which regional market experienced the largest decline in oil demand as a result of the Middle East conflict, and what unmeasured category of feedstocks contributed significantly to lost demand?

In [ ]:
if __name__ == "__main__":
    question = "Which regional market experienced the largest decline in oil demand as a result of the Middle East conflict, and what unmeasured category of feedstocks contributed significantly to lost demand?"
    rag_context = (
        "data have now been reported display a varied picture among regions and products. March’s \ncontraction was led by LPG/ethane, which, having accounted for more almost a quarter of \nnon-OECD demand growth in 2025, turned abruptly negative (-560 kb/d y-o-y). With Middle Eastern \npetrochemical operations still crippled by the war and the Strait of Hormuz closure depriving Asia of \nGulf feedstocks, the decline is set to deepen in 2Q26 (-1.5 mb/d). In regional terms, the Middle East \n(-780 kb/d y-o-y in 2Q26) and Asia (-710 kb/d) account for the bulk of the drop.  \n8.0\n8.5\n9.0\n9.5\n10.0\nJan Mar May Jul Sep Nov\nmb/d United States Gasoline Demand\n2019 2024 2025 2026\n3.8\n4.0\n4.2\n4.4\n4.6\nJan Mar May Jul Sep Nov\nmb/d United States Gasoil/Diesel Demand\n2019 2024 2025 2026')"
        "Oil Market Report Refining \nPAGE | 44  13 MAY 2026 \nIEA. All rights reserved. \nAsian Naphtha Demand Shrinks to Meet Supply \nPetrochemical producers are among the most price-sensitive consumers of oil. Faced at times with \nrazor-thin margins for high- cost naphtha-fed plants and with the potential to switch to cheaper \ncompeting feedstocks, such as propane, naphtha demand is notori ously fickle. Consequently, the \nloss of Middle East exports has not resulted in the knee-jerk price response or a substantial increase \nin cracks that other oil products have seen. Rather, the loss of naphtha exports to Asian markets has \nforced a rapid reduction in demand and transmitted the negative supply shock through petrochemical \nsupply chains globally. Thus far, naphtha consumption has dropped sharply across the Asia Pacific \nregion with year -on-year declines comparable to the contractions seen in the depths of the global \nfinancial crisis (GFC) of 2008-2009.')"
        "weeks. Global consumption is forecast to take its biggest hit in 2Q26, declining by 2.45 mb/d, with \nthe OECD down by 930 kb/d and the non-OECD shifting into a contraction of 1.5 mb/d y-o-y.  \nWithin the product spectrum, petrochemical feedstocks have been the most impacted by far. As the \nclosure of the Strait of Hormuz brought a sudden halt to the region’s exports, LPG/ethane and \nnaphtha account for around half of the forecast 2026 demand downgrade from pre-war levels, or \n700 kb/d on average, but with a much steeper correction in 2Q26. Jet/kerosene uptake has been \nthe other product most immediately affected, down by 210 kb/d from our pre-conflict estimate. \nConversely, the impact on road fuels has been relatively modest to date, with annual growth in gasoil \nand gasoline deliveries actually accelerating by around 300 kb/d each from February to March. This \npoints to consumers and businesses, wary of potential future supply disruptions, storing up on fuel')"
        "sectors. Demand in each country fell by 290 kb/d y-o-y in April, and we expect an overall regional \ndecline of 440 kb/d y-o-y during 2Q26, resulting in an annual decrease of 160 kb/d, half of which will \nbe for naphtha. \nReported Korean delivery data for March showed a 200 kb/d (more than 8%) decline, both m -o-m \nand y-o-y, in the first month after the outbreak of the Iran war. Demand fell across almost all product \ncategories, with petrochemical operations hit most. Korea is the world’s s econd-largest naphtha \nconsumer and a 4.8% fall for the feedstock is a harbinger of the pressure building on the sector in \nEast Asia and beyond. LPG use was down by 8% as its prices rose even more sharply than those \nfor naphtha. Gasoline and gasoil also marked declines, of around 5% each, with pump price rises \n-30%\n-20%\n-10%\n0%\n10%\n20%\n30%\n01-Feb 22-Feb 15-Mar 05-Apr 26-Apr\nFuel Sales/Deliveries, y-o-y change \nUK Sales France Deliveries\nSources: UK Department for Energy and Net Zero, Comité')]"

    )
    rag_answer = (
        "Based on the provided context:\n\n Regional Market with the Largest Decline:** In regional terms, the Middle East** accounts for the largest decline in oil demand (forecast to drop by 780 kb/d y-o-y in 2Q26, followed by Asia at -710 kb/d). On a broader grouping level, non-OECD demand sees the largest contraction (-1.5 mb/d y-o-y in 2Q26).\n Unmeasured Category of Feedstocks: The provided context does not mention any unmeasured category of feedstocks. It only specifies measured petrochemical feedstocks—primarily LPG/ethane and naphtha—as accounting for the major drop in demand."

    )

    evaluator = RAGEvaluator()
    report = evaluator.evaluate(
        question=question,
        context=rag_context,
        answer=rag_answer
    )

    print("--- RAG Evaluation Report ---")
    print(f"Overall RAG Score: {report.composite_score} / 1.0\n")
    print(f"Faithfulness Score: {report.faithfulness.faithfulness_score}")
    print(f"  Reasoning: {report.faithfulness.reasoning}")
    print("  Claims Breakdown:")
    for c in report.faithfulness.claims:
        status = "✓ Supported" if c.is_supported else "✗ Unsupported"
        print(f"    - [{status}] Claim: '{c.claim}' | Evidence: '{c.evidence}'")

    print(f"\nAnswer Relevance Score: {report.answer_relevance.relevance_score}")
    print(f"  Reasoning: {report.answer_relevance.reasoning}")

    print(f"\nContext Precision Score: {report.context_relevance.context_precision_score}")
    print(f"  Reasoning: {report.context_relevance.reasoning}")


--- RAG Evaluation Report ---
Overall RAG Score: 0.812 / 1.0

Faithfulness Score: 1.0
  Reasoning: All five extracted statements are directly supported by sentences in the provided context. The context explicitly mentions the magnitude of the decline for the Middle East and Asia, the non-OECD contraction, and highlights LPG/ethane and naphtha as the primary measured feedstocks, with no reference to any unmeasured categories. Therefore, the answer is fully faithful to the source.
  Claims Breakdown:
    - [✓ Supported] Claim: 'In regional terms, the Middle East accounts for the largest decline in oil demand (forecast to drop by 780 kb/d y-o-y in 2Q26).' | Evidence: 'In regional terms, the Middle East (-780 kb/d y-o-y in 2Q26) and Asia (-710 kb/d) account for the bulk of the drop.'
    - [✓ Supported] Claim: 'Asia accounts for the second largest decline at -710 kb/d y-o-y in 2Q26.' | Evidence: 'In regional terms, the Middle East (-780 kb/d y-o-y in 2Q26) and Asia (-710 kb/d) account for 

In [20]:
if __name__ == "__main__":    
    
    question = "Which regional market experienced the largest decline in oil demand as a result of the Middle East conflict, and what unmeasured category of feedstocks contributed significantly to lost demand?"
    graph_context = (
    
        "This query finds geopolitical events related to the Middle East and returns the regions they impact (as regional markets), along with any affected commodities (as feedstocks) if available."

    )
    graph_answer = (
        "Russia experienced the largest decline in oil demand, and the feedstocks contributing significantly were Crude and Commodity."
        
    )

    evaluator = RAGEvaluator()
    report = evaluator.evaluate(
        question=question,
        context=graph_context,
        answer=graph_answer
    )

    print("--- Graph RAG Evaluation Report ---")
    print(f"Overall Graph RAG Score: {report.composite_score} / 1.0\n")
    print(f"Faithfulness Score: {report.faithfulness.faithfulness_score}")
    print(f"  Reasoning: {report.faithfulness.reasoning}")
    print("  Claims Breakdown:")
    for c in report.faithfulness.claims:
        status = "✓ Supported" if c.is_supported else "✗ Unsupported"
        print(f"    - [{status}] Claim: '{c.claim}' | Evidence: '{c.evidence}'")

    print(f"\nAnswer Relevance Score: {report.answer_relevance.relevance_score}")
    print(f"  Reasoning: {report.answer_relevance.reasoning}")

    print(f"\nContext Precision Score: {report.context_relevance.context_precision_score}")
    print(f"  Reasoning: {report.context_relevance.reasoning}")

--- Graph RAG Evaluation Report ---
Overall Graph RAG Score: 0.24 / 1.0

Faithfulness Score: 0.0
  Reasoning: The provided context only describes a query that retrieves geopolitical events related to the Middle East and their impacted regions and commodities. It contains no information about Russia, oil demand declines, or specific feedstocks such as Crude and Commodity. Therefore, both extracted claims are unsupported by the context, resulting in a faithfulness score of 0.0.
  Claims Breakdown:
    - [✗ Unsupported] Claim: 'Russia experienced the largest decline in oil demand.' | Evidence: 'None'
    - [✗ Unsupported] Claim: 'The feedstocks contributing significantly were Crude and Commodity.' | Evidence: 'None'

Answer Relevance Score: 0.6
  Reasoning: The answer attempts to address both parts of the question by naming a regional market (Russia) and mentioning feedstock categories (Crude and Commodity). However, the question asks for a single "unmeasured category of feedstocks," and 

Question 10 

What entity serves as the primary bridge node between the petroleum supply chain and the electric power sector?

In [ ]:
if __name__ == "__main__":
    question = "What entity serves as the primary bridge node between the petroleum supply chain and the electric power sector?"
    rag_context = (
        "Crude oil product supplied Secondary supply chain  Tertiary stocks of natural gas liquids Components of Supply and Disposition  Balance of supply and disposition  Components of supply"

    )
    rag_answer = (
        "Based on the provided context, there is no mention of a primary bridge node or any specific entity serving as a bridge between the petroleum supply chain and the electric power sector. Therefore, the provided text does not contain the answer to this question."

    )

    evaluator = RAGEvaluator()
    report = evaluator.evaluate(
        question=question,
        context=rag_context,
        answer=rag_answer
    )

    print("--- RAG Evaluation Report ---")
    print(f"Overall RAG Score: {report.composite_score} / 1.0\n")
    print(f"Faithfulness Score: {report.faithfulness.faithfulness_score}")
    print(f"  Reasoning: {report.faithfulness.reasoning}")
    print("  Claims Breakdown:")
    for c in report.faithfulness.claims:
        status = "✓ Supported" if c.is_supported else "✗ Unsupported"
        print(f"    - [{status}] Claim: '{c.claim}' | Evidence: '{c.evidence}'")

    print(f"\nAnswer Relevance Score: {report.answer_relevance.relevance_score}")
    print(f"  Reasoning: {report.answer_relevance.reasoning}")

    print(f"\nContext Precision Score: {report.context_relevance.context_precision_score}")
    print(f"  Reasoning: {report.context_relevance.reasoning}")


--- RAG Evaluation Report ---
Overall RAG Score: 0.44 / 1.0

Faithfulness Score: 1.0
  Reasoning: Both extracted claims are directly supported by the context, which contains only supply chain terminology and no reference to a bridge node or electric power sector. Consequently, the answer correctly states that the context lacks the requested information, yielding a perfect faithfulness score.
  Claims Breakdown:
    - [✓ Supported] Claim: 'The provided context does not mention a primary bridge node or any specific entity serving as a bridge between the petroleum supply chain and the electric power sector.' | Evidence: 'Crude oil product supplied Secondary supply chain Tertiary stocks of natural gas liquids Components of Supply and Disposition Balance of supply and disposition Components of supply'
    - [✓ Supported] Claim: 'Therefore, the provided text does not contain the answer to this question.' | Evidence: 'Crude oil product supplied Secondary supply chain Tertiary stocks of natura

In [22]:
if __name__ == "__main__":    
    
    question = "What entity serves as the primary bridge node between the petroleum supply chain and the electric power sector?"
    graph_context = (
            
        "In the knowledge graph, the entity Refineries acts as a key bridge node with a strength score of 63."
    )
    graph_answer = (
        "The United States and the U.S. Energy Information Administration serve as the primary bridge nodes between the petroleum supply chain and the electric power sector."
        
    )

    evaluator = RAGEvaluator()
    report = evaluator.evaluate(
        question=question,
        context=graph_context,
        answer=graph_answer
    )

    print("--- Graph RAG Evaluation Report ---")
    print(f"Overall Graph RAG Score: {report.composite_score} / 1.0\n")
    print(f"Faithfulness Score: {report.faithfulness.faithfulness_score}")
    print(f"  Reasoning: {report.faithfulness.reasoning}")
    print("  Claims Breakdown:")
    for c in report.faithfulness.claims:
        status = "✓ Supported" if c.is_supported else "✗ Unsupported"
        print(f"    - [{status}] Claim: '{c.claim}' | Evidence: '{c.evidence}'")

    print(f"\nAnswer Relevance Score: {report.answer_relevance.relevance_score}")
    print(f"  Reasoning: {report.answer_relevance.reasoning}")

    print(f"\nContext Precision Score: {report.context_relevance.context_precision_score}")
    print(f"  Reasoning: {report.context_relevance.reasoning}")

--- Graph RAG Evaluation Report ---
Overall Graph RAG Score: 0.372 / 1.0

Faithfulness Score: 0.0
  Reasoning: The context only mentions that the entity Refineries is a key bridge node with a strength score of 63. It does not provide any information about the United States or the U.S. Energy Information Administration being bridge nodes, so both claims are unsupported.
  Claims Breakdown:
    - [✗ Unsupported] Claim: 'The United States serves as a primary bridge node between the petroleum supply chain and the electric power sector.' | Evidence: 'None'
    - [✗ Unsupported] Claim: 'The U.S. Energy Information Administration serves as a primary bridge node between the petroleum supply chain and the electric power sector.' | Evidence: 'None'

Answer Relevance Score: 0.45
  Reasoning: The answer attempts to address the question by naming entities that could act as a bridge between the petroleum supply chain and the electric power sector. However, the question asks for the primary bridge no

Question 11

How would a sudden collapse in global crude oil demand alter the relationships between OPEC production and OECD tax revenues?

In [ ]:
if __name__ == "__main__":

    question = "How would a sudden collapse in global crude oil demand alter the relationships between OPEC production and OECD tax revenues?"
    rag_context = (
            
        "average prices.Annual average revenues Tax Crude price Industry m argin E s t im a t e d  a n n u a l average OECD retail revenues E s t im a t e d  a n n u a laverage OPEC oil revenues %$bn/year OECD average OECD average Comparison of composite barrel (OECD average)/.null T a x C ru d e  p ric e"
    )
    rag_answer = (
        "Based on the provided context, there is no direct discussion or analysis of how a sudden collapse in global crude oil demand would alter the relationships between OPEC production and OECD tax revenues. However, we can infer the structural vulnerability of both components based on the data provided for the current relationship. Regarding **revenue disparity**, the estimated annual average OECD retail revenues (for 2020–2024) are **$2.571 trillion/year**, whereas annual average OPEC oil revenues are **$0.599 trillion/year**—resulting in a difference of **$1.972 trillion/year**. Regarding the **composite barrel share (OECD in 2023)**, **government tax** is $113.48, **crude price (to oil producers)** is $80.21, and **industry margin** is $52.73. As an analytical conclusion, in the event of a sudden demand collapse, **OPEC revenues** would likely fall precipitously because they are directly tied to the crude oil price ($80.21 of the composite barrel) and export volumes, while **OECD tax revenues** would also decline due to lower consumption volumes. However, because government taxes represent the largest single component of the composite barrel ($113.48)—and many retail fuel taxes in OECD countries are volume-based (fixed per gallon/liter) rather than value-based—the OECD tax mechanism provides a cushion, meaning OECD government revenues would likely prove more resilient per barrel than OPEC's crude revenues, further widening the existing $1.972 trillion revenue gap."
        
    )

    evaluator = RAGEvaluator()
    report = evaluator.evaluate(
        question=question,
        context=rag_context,
        answer=rag_answer
    )

    print("--- RAG Evaluation Report ---")
    print(f"Overall RAG Score: {report.composite_score} / 1.0\n")
    print(f"Faithfulness Score: {report.faithfulness.faithfulness_score}")
    print(f"  Reasoning: {report.faithfulness.reasoning}")
    print("  Claims Breakdown:")
    for c in report.faithfulness.claims:
        status = "✓ Supported" if c.is_supported else "✗ Unsupported"
        print(f"    - [{status}] Claim: '{c.claim}' | Evidence: '{c.evidence}'")

    print(f"\nAnswer Relevance Score: {report.answer_relevance.relevance_score}")
    print(f"  Reasoning: {report.answer_relevance.reasoning}")

    print(f"\nContext Precision Score: {report.context_relevance.context_precision_score}")
    print(f"  Reasoning: {report.context_relevance.reasoning}")


--- RAG Evaluation Report ---
Overall RAG Score: 0.36 / 1.0

Faithfulness Score: 0.0
  Reasoning: None of the factual claims in the answer are directly supported by the provided context, which lacks the specific numerical values and explanatory statements referenced. Therefore, all claims are marked as unsupported, resulting in a faithfulness score of 0.0.
  Claims Breakdown:
    - [✗ Unsupported] Claim: 'The estimated annual average OECD retail revenues (for 2020–2024) are $2.571 trillion/year.' | Evidence: 'None'
    - [✗ Unsupported] Claim: 'The estimated annual average OPEC oil revenues are $0.599 trillion/year.' | Evidence: 'None'
    - [✗ Unsupported] Claim: 'The difference between them is $1.972 trillion/year.' | Evidence: 'None'
    - [✗ Unsupported] Claim: 'In the composite barrel share (OECD in 2023), government tax is $113.48.' | Evidence: 'None'
    - [✗ Unsupported] Claim: 'In the composite barrel share, crude price (to oil producers) is $80.21.' | Evidence: 'None'
    - [

In [24]:
if __name__ == "__main__":    
    
    question = "How would a sudden collapse in global crude oil demand alter the relationships between OPEC production and OECD tax revenues?"
    graph_context = (
            
        "This dataset details graph paths connecting OPEC's supply actions and market forecasts to fuel tax policies, tracing relationships through global oil demand across key regions like China, India, and the United States, alongside intermediate drivers such as pump prices, Middle East conflict, and economic forecasts."
    )
    graph_answer = (
        "The direct incoming and outgoing relationships centered on the Permian Region node are:\n\n**Outgoing Relationships:**\n*   Permian Region is **LOCATED_IN** the United States.\n*   Permian Region is **LOCATED_IN** the Lower 48 States.\n*   Permian Region **AFFECTS_SUPPLY_OF** Total Primary Supply.\n*   Permian Region **PRODUCES** Natural Gas.\n\n**Incoming Relationships:**\n*   Cumulative Drilled But Uncompleted Wells is **LOCATED_IN** the Permian Region.\n*   New Wells Drilled is **MEASURED_IN** the Permian Region.\n*   New Wells Drilled Per Rig is **MEASURED_IN** the Permian Region.\n*   U.S. Energy Information Administration **FORECASTS_OUTLOOK** the Permian Region.\n*   Crude Oil is **PRODUCED_IN** the Permian Region.\n*   Crude Oil Production From Newly Completed Wells is **PRODUCED_IN** the Permian Region."
        
    )

    evaluator = RAGEvaluator()
    report = evaluator.evaluate(
        question=question,
        context=graph_context,
        answer=graph_answer
    )

    print("--- Graph RAG Evaluation Report ---")
    print(f"Overall Graph RAG Score: {report.composite_score} / 1.0\n")
    print(f"Faithfulness Score: {report.faithfulness.faithfulness_score}")
    print(f"  Reasoning: {report.faithfulness.reasoning}")
    print("  Claims Breakdown:")
    for c in report.faithfulness.claims:
        status = "✓ Supported" if c.is_supported else "✗ Unsupported"
        print(f"    - [{status}] Claim: '{c.claim}' | Evidence: '{c.evidence}'")

    print(f"\nAnswer Relevance Score: {report.answer_relevance.relevance_score}")
    print(f"  Reasoning: {report.answer_relevance.reasoning}")

    print(f"\nContext Precision Score: {report.context_relevance.context_precision_score}")
    print(f"  Reasoning: {report.context_relevance.reasoning}")

--- Graph RAG Evaluation Report ---
Overall Graph RAG Score: 0.144 / 1.0

Faithfulness Score: 0.0
  Reasoning: The provided context only describes a high-level dataset about OPEC, global oil demand, and related drivers, and does not contain any information about the Permian Region or the specific relationships listed. Consequently, none of the extracted claims are supported by the context.
  Claims Breakdown:
    - [✗ Unsupported] Claim: 'Permian Region is LOCATED_IN the United States.' | Evidence: 'None'
    - [✗ Unsupported] Claim: 'Permian Region is LOCATED_IN the Lower 48 States.' | Evidence: 'None'
    - [✗ Unsupported] Claim: 'Permian Region AFFECTS_SUPPLY_OF Total Primary Supply.' | Evidence: 'None'
    - [✗ Unsupported] Claim: 'Permian Region PRODUCES Natural Gas.' | Evidence: 'None'
    - [✗ Unsupported] Claim: 'Cumulative Drilled But Uncompleted Wells is LOCATED_IN the Permian Region.' | Evidence: 'None'
    - [✗ Unsupported] Claim: 'New Wells Drilled is MEASURED_IN the Permi

Question 12 

List all incoming and outgoing connections for the Brent pricing benchmark node.

In [ ]:
if __name__ == "__main__":

    question = "List all incoming and outgoing connections for the Brent pricing benchmark node."
    rag_context = (
            
        "Americas Europe Asia Oceania Total OECD NON-OECD SUPPLY Eurasia Europe China  Other Asia Latin America Middle East "
    )
    rag_answer = (
        "Based on the provided context, there is no mention of the Brent pricing benchmark node or any incoming and outgoing connections associated with it. Therefore, I do not have the information to answer this question."

    )

    evaluator = RAGEvaluator()
    report = evaluator.evaluate(
        question=question,
        context=rag_context,
        answer=rag_answer
    )

    print("--- RAG Evaluation Report ---")
    print(f"Overall RAG Score: {report.composite_score} / 1.0\n")
    print(f"Faithfulness Score: {report.faithfulness.faithfulness_score}")
    print(f"  Reasoning: {report.faithfulness.reasoning}")
    print("  Claims Breakdown:")
    for c in report.faithfulness.claims:
        status = "✓ Supported" if c.is_supported else "✗ Unsupported"
        print(f"    - [{status}] Claim: '{c.claim}' | Evidence: '{c.evidence}'")

    print(f"\nAnswer Relevance Score: {report.answer_relevance.relevance_score}")
    print(f"  Reasoning: {report.answer_relevance.reasoning}")

    print(f"\nContext Precision Score: {report.context_relevance.context_precision_score}")
    print(f"  Reasoning: {report.context_relevance.reasoning}")


--- RAG Evaluation Report ---
Overall RAG Score: 0.4 / 1.0

Faithfulness Score: 1.0
  Reasoning: All three extracted statements are directly supported by the context, which contains only a list of region names and no reference to the Brent pricing benchmark node or its connections. Therefore, the answer correctly reflects the lack of relevant information.
  Claims Breakdown:
    - [✓ Supported] Claim: 'The provided context does not mention the Brent pricing benchmark node.' | Evidence: 'Americas Europe Asia Oceania Total OECD NON-OECD SUPPLY Eurasia Europe China  Other Asia Latin America Middle East'
    - [✓ Supported] Claim: 'The provided context does not mention any incoming and outgoing connections associated with the Brent pricing benchmark node.' | Evidence: 'Americas Europe Asia Oceania Total OECD NON-OECD SUPPLY Eurasia Europe China  Other Asia Latin America Middle East'
    - [✓ Supported] Claim: 'I do not have the information to answer this question.' | Evidence: 'Americas Eu

In [30]:
if __name__ == "__main__":    
    
    question = "List all incoming and outgoing connections for the Brent pricing benchmark node."
    graph_context = (
            
        "Brent is located in North Sea. Brent is located in Rotterdam. Brent is measured in Us$/B. Ice operates Brent. Opec forecasts the outlook for Brent. Brent is produced in North Sea. Brent is produced in Northern Europe. Brent impacts the price of North Sea Dated. Managed Money Net Long Positions impacts the price of Brent."
    )
    graph_answer = (
        "The incoming and outgoing connections for the Brent pricing benchmark node are outgoing connections (from Brent), including LOCATED_IN North Sea, Rotterdam, MEASURED_IN Us$/B, PRODUCED_IN North Sea, Northern Europe, and IMPACTS_PRICE North Sea Dated, as well as incoming connections (to Brent), including OPERATES Ice, FORECASTS_OUTLOOK Opec, and IMPACTS_PRICE Managed Money Net Long Positions."

    )

    evaluator = RAGEvaluator()
    report = evaluator.evaluate(
        question=question,
        context=graph_context,
        answer=graph_answer
    )

    print("--- Graph RAG Evaluation Report ---")
    print(f"Overall Graph RAG Score: {report.composite_score} / 1.0\n")
    print(f"Faithfulness Score: {report.faithfulness.faithfulness_score}")
    print(f"  Reasoning: {report.faithfulness.reasoning}")
    print("  Claims Breakdown:")
    for c in report.faithfulness.claims:
        status = "✓ Supported" if c.is_supported else "✗ Unsupported"
        print(f"    - [{status}] Claim: '{c.claim}' | Evidence: '{c.evidence}'")

    print(f"\nAnswer Relevance Score: {report.answer_relevance.relevance_score}")
    print(f"  Reasoning: {report.answer_relevance.reasoning}")

    print(f"\nContext Precision Score: {report.context_relevance.context_precision_score}")
    print(f"  Reasoning: {report.context_relevance.reasoning}")

--- Graph RAG Evaluation Report ---
Overall Graph RAG Score: 0.79 / 1.0

Faithfulness Score: 1.0
  Reasoning: All extracted claims directly correspond to statements present in the provided context, and each claim is supported by a specific sentence from the context.
  Claims Breakdown:
    - [✓ Supported] Claim: 'Brent has an outgoing connection LOCATED_IN North Sea.' | Evidence: 'Brent is located in North Sea.'
    - [✓ Supported] Claim: 'Brent has an outgoing connection LOCATED_IN Rotterdam.' | Evidence: 'Brent is located in Rotterdam.'
    - [✓ Supported] Claim: 'Brent has an outgoing connection MEASURED_IN Us$/B.' | Evidence: 'Brent is measured in Us$/B.'
    - [✓ Supported] Claim: 'Brent has an outgoing connection PRODUCED_IN North Sea.' | Evidence: 'Brent is produced in North Sea.'
    - [✓ Supported] Claim: 'Brent has an outgoing connection PRODUCED_IN Northern Europe.' | Evidence: 'Brent is produced in Northern Europe.'
    - [✓ Supported] Claim: 'Brent has an outgoing connecti